# Milestone 1 Objective

Create the clean project foundation and test only the standalone Orchestrator Agent. This milestone does not run API, UI, performance, RAG, MCP, Selenium, Locust, dashboard, repository cloning, or real LLM calls.

## Why Start With The Orchestrator Agent Alone

The Orchestrator is the entry point of the future multi-agent workflow. It validates the user request, chooses which future agents should run, records risks, and saves a structured JSON result. Testing it alone keeps the first milestone deterministic and easy to verify.

## LangGraph Mini Workflow

```text
START -> orchestrator -> END
```

## State As Shared Workflow Memory

LangGraph State is the shared memory of the workflow. Each node reads from the State and returns partial updates. Future agents will add project information, retrieved context, test plans, execution results, bug analysis, and reports.

## Security Note

API keys must be stored in a local `.env` file, not in code or notebooks. This notebook only displays boolean readiness flags and never prints secret values.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from test_auto.agents.orchestrator import run_orchestrator_alone

In [2]:
repo_url = "https://github.com/example/todo-app"
target_url = "http://localhost:8000"
user_preferences = {
    "test_types": ["api", "ui"],
    "execution_mode": "parallel",
    "max_duration_minutes": 5,
}

In [3]:
orchestrator_result = run_orchestrator_alone(
    repo_url=repo_url,
    target_url=target_url,
    user_preferences=user_preferences,
)

In [4]:
import json

print(json.dumps(orchestrator_result, indent=2))

{
  "run_id": "run_20260520T083943Z_e13082ea",
  "selected_agents": [
    "repository_analyzer",
    "rag",
    "test_planner",
    "api",
    "ui",
    "bug",
    "report"
  ],
  "orchestrator_decision": {
    "run_id": "run_20260520T083943Z_e13082ea",
    "selected_agents": [
      "repository_analyzer",
      "rag",
      "test_planner",
      "api",
      "ui",
      "bug",
      "report"
    ],
    "execution_mode": "parallel",
    "reasoning_summary": "Selected future agents from requested test types using deterministic milestone-1 rules. No testing agent was executed.",
    "risks": [],
    "next_node": "repository_analyzer"
  },
  "agent_logs": [
    {
      "agent": "orchestrator",
      "timestamp": "2026-05-20T08:39:43.092695+00:00",
      "status": "success",
      "duration_seconds": 0.00020349997794255614,
      "summary": {
        "total_tests": 0,
        "passed": 0,
        "failed": 0,
        "pass_rate": 0.0
      },
      "tests": [],
      "anomalies": [],
     

In [5]:
from test_auto.graph.workflow import run_workflow

In [6]:
initial_state = {
    "run_id": "",
    "repo_url": repo_url,
    "target_url": target_url,
    "user_preferences": user_preferences,
    "selected_agents": [],
    "orchestrator_decision": {},
    "errors": [],
    "agent_logs": [],
}

final_state = run_workflow(initial_state)

In [7]:
print(json.dumps(final_state, indent=2))

{
  "run_id": "run_20260520T083945Z_9486af76",
  "repo_url": "https://github.com/example/todo-app",
  "target_url": "http://localhost:8000",
  "user_preferences": {
    "test_types": [
      "api",
      "ui"
    ],
    "execution_mode": "parallel",
    "max_duration_minutes": 5
  },
  "selected_agents": [
    "repository_analyzer",
    "rag",
    "test_planner",
    "api",
    "ui",
    "bug",
    "report"
  ],
  "orchestrator_decision": {
    "run_id": "run_20260520T083945Z_9486af76",
    "selected_agents": [
      "repository_analyzer",
      "rag",
      "test_planner",
      "api",
      "ui",
      "bug",
      "report"
    ],
    "execution_mode": "parallel",
    "reasoning_summary": "Selected future agents from requested test types using deterministic milestone-1 rules. No testing agent was executed.",
    "risks": [],
    "next_node": "repository_analyzer"
  },
  "errors": [
    {
      "agent": "repo_analyzer",
      "field": "repo_path",
      "message": "Command '['git', 'c

In [8]:
from test_auto.shared.secrets import get_llm_config

print(json.dumps(get_llm_config(), indent=2))

{
  "provider": "groq",
  "has_groq_key": false,
  "has_mistral_key": false,
  "groq_model": null,
  "mistral_model": null
}
